# panel based grid

In [ ]:
case_grid =r'D:\shanghai workshop\vector\visual.shp'

# clip
# 00 lst

In [31]:
import rasterio
import geopandas as gpd
from rasterio.mask import mask
import os

def split_raster_by_grid(original_raster, grid_path, output_dir):
    # 读取矢量网格
    grid = gpd.read_file(grid_path)

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(original_raster) as src:
        raster_crs = src.crs
        print("📌 原始栅格坐标系:", raster_crs)

        # 坐标系统一
        if grid.crs != raster_crs:
            print("🔄 正在将矢量网格投影为与栅格一致的坐标系...")
            grid = grid.to_crs(raster_crs)

        for _, row in grid.iterrows():
            tile_id = row["ID"]  # 使用 grid 中的 ID 字段
            geom = [row.geometry]

            try:
                out_image, out_transform = mask(src, geom, crop=True)
            except Exception as e:
                print(f"⚠️ 跳过 ID={tile_id}，裁剪失败: {e}")
                continue

            out_meta = src.meta.copy()
            out_meta.update({
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs
            })

            out_path = os.path.join(output_dir, f"lst_tile_{tile_id}.tif")
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_image)

            print(f"✅ 已保存切块: {out_path}")

# 示例调用
split_raster_by_grid(

    original_raster=r'D:\shanghai workshop\lst\original\final_lst\clipped_20220814_B10_cleaned_LST.tif',
    grid_path=r'D:\shanghai workshop\vector\visual.shp',
    output_dir=r"D:\shanghai workshop\raster\visual\lst_merge"
)

📌 原始栅格坐标系: EPSG:32651
🔄 正在将矢量网格投影为与栅格一致的坐标系...
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51737.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51738.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51739.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51740.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51741.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51742.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51743.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51744.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51745.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51746.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51747.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51748.tif
✅ 已保存切块: D:\shanghai workshop\raster\visual\lst_merge\lst_tile_51749.tif
✅ 已保

# 01